# San Francisco Assessor Roll Data Analysis

This notebook analyzes the San Francisco historical secured property tax roll data for visualization and modeling. The main goal is to turn a large administrative dataset into a clean, interpretable geospatial property dataset that can later support visualization, neighborhood comparison, and price-related modeling.

## What this notebook does
1. Loads the raw assessor roll and applies an explicit schema so columns keep the intended data types.
2. Parses the geometry column and converts the table into a geospatially usable structure.
3. Checks whether small assessment components such as fixtures and personal property are material enough to include.
4. Builds a working measure of total assessed value.
5. Summarizes the composition of San Francisco's property stock by broad property type.
6. Examines how property counts and assessed values evolve over time.
7. Focuses on single-family homes and condominiums, since these are the most relevant categories for later housing-focused work.
8. Compares neighborhoods using assessed value, assessed value per square foot, building age, and recent transaction activity.

## Why this matters
Raw assessor data is rich but not immediately analysis-ready. Important fields require cleaning, category consolidation, geospatial parsing, and careful interpretation. This notebook establishes a transparent foundation so that later modeling work is based on clearly documented assumptions rather than opaque preprocessing steps.

## Important interpretation note
This dataset contains **assessed values**, not a direct market sale-price series. In California, Proposition 13 strongly affects how assessed values evolve over time. That means the notebook treats assessed value as a useful proxy for long-run property value, while also separately examining properties with more recent transactions to get closer to current market conditions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geopandas as gpd
from shapely import wkt
import contextily as cx
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from IPython.display import display


In [ ]:
# ------------------- #
# style and format    #
# ------------------- #

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 130,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.title_fontsize": 10,
})

def fmt_int(value):
    return f"{int(value):,}" if pd.notna(value) else "—"

def fmt_pct(value, digits=1):
    return f"{value:.{digits}f}%" if pd.notna(value) else "—"

def fmt_usd(value, digits=0):
    return f"${value:,.{digits}f}" if pd.notna(value) else "—"

def fmt_usd_m(value, digits=3):
    return f"${value:,.{digits}f}M" if pd.notna(value) else "—"

def fmt_usd_b(value, digits=3):
    return f"${value:,.{digits}f}B" if pd.notna(value) else "—"

def fmt_psf(value, digits=0):
    return f"${value:,.{digits}f} / sq ft" if pd.notna(value) else "—"

def fmt_years(value, digits=0):
    return f"{value:,.{digits}f} years" if pd.notna(value) else "—"

def style_table(df, caption=None, formatters=None):
    styler = (
        df.style
        .format(formatters or {})
        .set_properties(**{
            "text-align": "left",
            "font-size": "11pt",
            "padding": "6px 8px",
            "white-space": "nowrap",
        })
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "left"), ("font-size", "11pt"), ("padding", "6px 8px")]},
            {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left"), ("font-size", "12pt"), ("font-weight", "bold"), ("padding", "0 0 8px 0")]},
        ])
    )
    if caption:
        styler = styler.set_caption(caption)
    return styler

def clean_year_index(index_like):
    return pd.Index(pd.to_datetime(index_like).year.astype(str), name="Roll Year")

def apply_standard_axis_style(ax, y_is_currency=False, currency_scale=1, y_suffix="", rotate_x=45):
    ax.grid(axis="y", alpha=0.2, linewidth=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", rotation=rotate_x)
    if y_is_currency:
        if currency_scale == 1_000_000_000:
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x/1_000_000_000:,.0f}B"))
        elif currency_scale == 1_000_000:
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x/1_000_000:,.1f}M"))
        else:
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x:,.0f}{y_suffix}"))
    else:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:,.0f}{y_suffix}"))


In [ ]:
# ---------------- #
# PARAMETER SET UP #
# ---------------- #

# You should download the data from: https://data.sfgov.org/d/wv5m-vpq2
# The full dataset is over 1G in size.
INPUT_DATA_CSV = 'raw_data/wv5m-vpq2_version_10715.csv'

In [ ]:
# when importing, make sure the data types for the fields are set properly
field_dtype = {
    'closed_roll_year': 'Int64',
    'property_location': 'string',
    'parcel_number': 'string',
    'block': 'string',
    'lot': 'string',
    'volume_number': 'Int64',
    'use_code': 'string',
    'use_definition': 'string',
    'property_class_code': 'string',
    'property_class_code_definition': 'string',
    'year_property_built': 'Int64',
    'number_of_bathrooms': 'Int64',
    'number_of_bedrooms': 'Int64',
    'number_of_rooms': 'Int64',
    'number_of_stories': 'Int64',
    'number_of_units': 'Int64',
    'zoning_code': 'string',
    'construction_type': 'string',
    'lot_code': 'string',
    'tax_rate_area_code': 'string',
    'exemption_code': 'string',
    'exemption_code_definition': 'string',
    'status_code': 'string',
    'assessor_neighborhood_district': 'string',
    'assessor_neighborhood_code': 'string',
    'assessor_neighborhood': 'string',
    'supervisor_district': 'string',
    'supervisor_district_2012': 'string',
    'analysis_neighborhood': 'string',
    'row_id': 'string'
 }


df = pd.read_csv(INPUT_DATA_CSV, dtype=field_dtype, parse_dates=["current_sales_date", "data_as_of", "data_loaded_at"])

In [ ]:
# format the year properly
df['closed_roll_year'] = pd.to_datetime(df['closed_roll_year'], format="%Y")

# clean up the geometry
df.dropna(subset=['the_geom'], inplace=True)
df['geometry'] = df['the_geom'].apply(wkt.loads)
df.drop(['the_geom'], axis=1, inplace=True)

In [ ]:
############
# OPTIONAL #
############
# extract and store the definitions of various codes used in the dataset
# so that we can drop certain columns without losing information

# definition_use_code = df[['use_code', 'use_definition']].drop_duplicates().dropna().sort_values('use_code')
# definition_use_code.to_csv('definition/definition_use_code.csv', index=False)

# definition_property_class_code = df[['use_code', 'property_class_code', 'property_class_code_definition']].drop_duplicates().dropna().sort_values(['use_code','property_class_code'])
# definition_property_class_code.to_csv('definition/definition_property_class_code.csv', index=False)

# definition_exemption_code = df[['exemption_code', 'exemption_code_definition']].drop_duplicates().dropna().sort_values('exemption_code')
# definition_exemption_code.to_csv('definition/definition_exemption_code.csv', index=False)

# definition_assessor_neighborhood_code = df[['assessor_neighborhood_district', 'assessor_neighborhood_code', 'assessor_neighborhood']].drop_duplicates().dropna().sort_values('assessor_neighborhood_code')
# definition_assessor_neighborhood_code.to_csv('definition/definition_assessor_neighborhood_code.csv', index=False)

## 1. Validate whether fixture and personal-property assessments can be ignored

Before constructing a simplified total value measure, it is useful to check whether two smaller assessed components—**fixtures** and **personal property**—are common or economically meaningful in this dataset.

The purpose of this step is methodological: if these fields are almost always zero or trivial, excluding them makes the downstream analysis cleaner without materially changing conclusions. That helps keep the value measure focused on the two core components of real estate taxation: **land value** and **improvement value**.

In [ ]:
summary = pd.DataFrame({
    "Share of Records with Non-Zero Value": [
        (df["assessed_fixtures_value"] > 0).mean() * 100,
        (df["assessed_personal_property_value"] > 0).mean() * 100,
    ],
    "Median Assessed Amount (USD)": [
        df["assessed_fixtures_value"].median(),
        df["assessed_personal_property_value"].median(),
    ],
}, index=["Fixtures", "Personal property"])

display(
    style_table(
        summary,
        caption="Materiality check for fixture and personal-property assessments",
        formatters={
            "Share of Records with Non-Zero Value": lambda x: fmt_pct(x, 2),
            "Median Assessed Amount (USD)": fmt_usd,
        },
    )
)


#### Observation and decision

The auxiliary assessment fields are present, but they are economically minor at the citywide level.

- Only **0.64%** of records have a non-zero **fixture** assessment, and only **4.68%** have a non-zero **personal-property** assessment.
- The **median assessed amount is $0** for both fields, which means the typical parcel does not carry either component.
- For broad spatial and cross-sectional analysis, excluding these fields introduces very little distortion while keeping the valuation logic transparent.

**Modeling choice used throughout the notebook:** total assessed value is defined as **land value + improvement value**.


## 2. Construct total assessed value

This step creates the core value variable used throughout the notebook:

- **`assessed_total`** = land assessment + improvement assessment
- **`log10_assessed_total`** = log10-scaled version used for better distributional behavior when needed

The raw assessed-value distribution is highly skewed, so keeping both the original scale and a log10 version is useful. The original scale is easier to interpret in dollars, while the log10 scale is often better for modeling and visualization when extreme values are present. A pseudo count of 1 is added to avoid applying log10 to number 0.

In [ ]:
df["assessed_total"] = df["assessed_land_value"].fillna(0) + df["assessed_improvement_value"].fillna(0)
df["log10_assessed_total"] = np.log10(df["assessed_total"] + 1)

## 3. Analysis: number and proportion of property types

The assessor roll contains many detailed property classifications. For a citywide structural view, the notebook groups them into broader categories such as single-family residential, multi-family residential, retail, office, hotel, industrial, and public/vacant land.

This section answers two basic but important questions:

1. **What kinds of properties make up San Francisco's built environment?**
2. **How has that composition changed over time?**

### Category notes
- **Single Family Residential (SRES) - '1 Single Family (SRES)':** mainly dwellings, condominiums, and town houses.
- **Multi Family Residential (MRES) - '2 Multi Family (MRES)':** mainly flats, duplexes, apartments, and mixed flat/store buildings.
- **Commercial Retail (COMR) - '3 Retail (COMR)':** mainly commercial stores and store units in condominiums.
- **Commercial Office (COMO) - '4 Office (COMO)':** mainly office buildings, government offices, and office units in condominiums.
- **Commercial Hotel (COMH) - '5 Hotel (COMH)':** mainly hotels and motels.
- **Commercial Misc (COMM) - '6 Timeshare (COMM)':** mainly timeshare properties.
- **Industrial (IND) - '7 Industrial (IND)':** mainly industrial and warehouse properties.
- **Government (GOVT) - '8 Vacant Gov (GOVT)':** mainly government-owned vacant land.
- **Miscellaneous/Mixed-Use (MISC) - '9 Vacant Public (MISC)':** mainly public-use or residential vacant lots.

Collapsing the detailed use codes into a smaller set of analytically meaningful groups makes later charts much easier to read and compare.

In [ ]:
property_type_lookup = (
    df[["use_code", "use_definition"]]
    .drop_duplicates()
    .dropna()
    .sort_values("use_code", ascending=False)
    .rename(columns={
        "use_code": "Use Code",
        "use_definition": "Property Type Description",
    })
    .reset_index(drop=True)
)

display(style_table(property_type_lookup, caption="Property type codes appearing in the assessor roll"))


In [ ]:
# rename to terms easier to understand
use_code_maps = {'SRES':'1 Single Family (SRES)',
                 'MRES':'2 Multi Family (MRES)',
                 'COMR':'3 Retail (COMR)',
                 'COMO':'4 Office (COMO)', 
                 'COMH':'5 Hotel (COMH)', 
                 'COMM':'6 Timeshare (COMM)', 
                 'IND':'7 Industrial (IND)', 
                 'GOVT':'8 Vacant Gov (GOVT)', 
                 'MISC':'9 Vacant Public (MISC)'}

### 3(A) Where are the various property types located in San Francisco?

A citywide location map helps connect property use to urban structure. This is important because counts alone do not show whether a category is diffuse, corridor-based, waterfront-oriented, or concentrated in the downtown core.

By plotting the 2024 property records spatially, we can compare the residential fabric of western and southern neighborhoods with the more commercial and industrial concentration in central and eastern San Francisco.

In [ ]:
df_2024 = df.loc[df['closed_roll_year']=='2024-01-01'].copy()
df_2024['use_code'] = df_2024['use_code'].map(use_code_maps)

## add a bit more information
df_2024['assessor_neighborhood'] = df_2024['assessor_neighborhood_code'] + ' ' + df_2024['assessor_neighborhood'] 

In [ ]:
gdf_2024 = gpd.GeoDataFrame(df_2024[['use_code', 'geometry']], geometry='geometry', crs="EPSG:4326")

data_3857 = gdf_2024.to_crs(epsg=3857)
color_mapping = {'1 Single Family (SRES)':'C0', 
                 '2 Multi Family (MRES)':'C1', 
                 '3 Retail (COMR)':'C2',
                 '4 Office (COMO)':'C3', 
                 '5 Hotel (COMH)':'C4', 
                 '6 Timeshare (COMM)':'C5', 
                 '7 Industrial (IND)':'C6', 
                 '8 Vacant Gov (GOVT)':'C7',
                 '9 Vacant Public (MISC)':'C8'}

fig, ax = plt.subplots(figsize=(9, 10))

data_3857.plot(
    ax=ax,
    color=data_3857["use_code"].map(color_mapping),
    markersize=2,
    alpha=0.9,
        zorder=3,
)

cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, attribution=False)

ax.set_title('Location of Various Property Types in San Francisco', fontsize=14)
ax.set_axis_off()

# Zoom to the building extent with a small margin so the map fills the page neatly.
xmin, ymin, xmax, ymax = data_3857.total_bounds
xpad = (xmax - xmin) * 0.05
ypad = (ymax - ymin) * 0.05
ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)

legend_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='w',
        label=label,
        markerfacecolor=color,
        markersize=10
    )
    for label, color in color_mapping.items()
]

ax.legend(handles=legend_handles, title='Property Type', loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# For each property type, identify the top 5 neighborhoods by record count in 2024.
top5 = (
    df_2024
    .value_counts(["use_code", "assessor_neighborhood"])
    .groupby(level=0)
    .head(5)
    .reset_index(name="property_count")
)

top5["rank"] = top5.groupby("use_code").cumcount() + 1

top5_pivot = top5.pivot(index="rank", columns="use_code", values="assessor_neighborhood")
top5_pivot.index.name = "Rank"
top5_pivot.columns.name = "Property Type"

display(style_table(top5_pivot, caption="Top 5 neighborhoods by property count within each property type"))


#### Observations

The neighborhood concentration pattern is highly consistent with San Francisco's built form.

- **Single-family homes** are most common in outer and hill-oriented residential neighborhoods such as **Bernal Heights, Parkside, Central Sunset, and Excelsior**.
- **Multi-family housing** is densest in urban neighborhoods such as **Inner Mission, Central Richmond, Noe Valley, Inner Richmond, and Inner Sunset**, which fits the city's corridor-style apartment geography.
- **Office and hotel uses** cluster in the historic downtown core—especially **Financial District North, Union Square, Financial District South, South of Market, and Civic Center**—while **industrial** parcels concentrate in **Bayview, Potrero Hill, South of Market, and Dogpatch/Central Waterfront**.
- The vacant-government footprint is concentrated in large special-purpose land areas such as **Treasure Island / Yerba Buena Island, Hunters Point, Bayview, Dogpatch, and Mission Bay**.

This is an important framing result: the assessor roll is not just a value dataset, but also a compact map of San Francisco's economic structure.


Source: https://thefrontsteps.com/san-francisco-real-estate-district-maps/  

![SF Real Estate District](https://i0.wp.com/thefrontsteps.com/wp-content/uploads/2021/12/San-Francisco-Real-Estate-Districts-Map.jpg?w=701ssl=1)

### 3(B) How do the various property types change over time?

This section moves from a single-year map to a historical view. Looking across roll years helps answer whether the city's property stock is stable or whether some categories expand, shrink, or respond to major shocks.

Two complementary views are useful:
- **absolute counts**, which show how many properties fall in each group;
- **shares of total properties**, which show the long-run composition of the city's property base.

In [ ]:
use_code_year = df.groupby(["closed_roll_year", "use_code"]).size().unstack(fill_value=0)
use_code_year.columns = use_code_year.columns.map(use_code_maps)
use_code_year = use_code_year[use_code_year.columns.sort_values()]
use_code_year.index = clean_year_index(use_code_year.index)

display(
    style_table(
        use_code_year.reset_index(),
        caption="Property counts by type and roll year",
        formatters={col: fmt_int for col in use_code_year.columns},
    )
)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- LEFT: all property types ---
use_code_year.plot(ax=ax1, legend=False, linewidth=2)
ax1.set_title("Property Count by Type and Roll Year")
ax1.set_xlabel("Roll Year")
ax1.set_ylabel("Number of properties")
apply_standard_axis_style(ax1)

# --- RIGHT: non-residential categories only ---
subset = use_code_year.drop(["1 Single Family (SRES)", "2 Multi Family (MRES)"], axis=1)
subset.plot(ax=ax2, color=['C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8'], linewidth=2)
ax2.set_title("Non-Residential Property Count by Roll Year")
ax2.set_xlabel("Roll Year")
ax2.set_ylabel("Number of properties")
apply_standard_axis_style(ax2)

# --- LEGEND ---
handles, labels = ax2.get_legend_handles_labels()
extra_handles = [Line2D([0], [0], color="C0"), Line2D([0], [0], color="C1")]
extra_labels = ["1 Single Family (SRES)", "2 Multi Family (MRES)"]

ax2.legend(
    extra_handles + handles,
    extra_labels + labels,
    title="Property type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)

plt.tight_layout()
plt.show()


In [ ]:
mean_props = use_code_year.div(use_code_year.sum(axis=1), axis=0).mean()
# mean_props = mean_props.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(5,5))
wedges, texts, autotexts = ax.pie(
    mean_props,
    autopct='%1.1f%%',
    labels=None
)
ax.set_title("Proportion of Various Property Types")
# custom legend: first 2 only
ax.legend(
    wedges,
    mean_props.index,
    bbox_to_anchor=(1, 1),
    title='Property Type'
)
plt.tight_layout()
plt.show()

#### Observations

The roll composition is remarkably stable, but the long-run trend still reveals a few meaningful structural shifts.

- In **2024**, the roll contains **154,886 single-family** records and **36,320 multi-family** records, so residential properties dominate the dataset by count.
- **Single-family** records fell from **140,514 in 2010** to **132,843 in 2011**, a drop of about **5.5%**, before recovering and reaching a new high by 2024.
- **Multi-family** records drifted slightly downward over time, from **37,941 in 2007** to **36,320 in 2024**, a decline of roughly **4.3%**.
- **Timeshare** properties decline materially after 2020, which may reflect pandemic-era stress on tourism-oriented property uses.
- The small specialist categories are far more cyclical. That matters because a change of a few hundred records can look large in percentage terms even when it is small relative to the whole tax base.

The main takeaway is that San Francisco's property inventory changes slowly. Most of the action in this notebook therefore comes not from dramatic shifts in counts, but from **where value is concentrated**, **how assessments differ across neighborhoods**, and **which parts of the market are actually transacting**.


## 4. Analysis: assessed values of various property types

After understanding **how many** properties exist in each category, the next question is **how much value** each category represents.

This section looks at:
- the **median assessed value** for an individual property in each category; and
- the **total assessed value** contributed by that category across the city.

These two views answer different questions:
- Median value helps describe the typical scale of a property type.
- Total value helps describe the category's contribution to the city's tax base.

**Data note:** the 2014 roll appears to have missing or unusable assessment values, so that year is excluded from the value trend analysis.

In [ ]:
# Median assessed value per property, shown in millions of USD for readability.
median_value_year = (
    df.loc[df["closed_roll_year"] != "2014-01-01"]
    .groupby(["closed_roll_year", "use_code"])["assessed_total"]
    .median()
    .unstack()
    / 1_000_000
)
median_value_year.columns = median_value_year.columns.map(use_code_maps)
median_value_year = median_value_year[median_value_year.columns.sort_values()].round(3)
median_value_year.index = clean_year_index(median_value_year.index)

display(
    style_table(
        median_value_year.reset_index(),
        caption="Median assessed value by property type and roll year",
        formatters={col: fmt_usd_m for col in median_value_year.columns},
    )
)


In [ ]:
# Total assessed value, shown in billions of USD for readability.
total_value_year = (
    df.loc[df["closed_roll_year"] != "2014-01-01"]
    .groupby(["closed_roll_year", "use_code"])["assessed_total"]
    .sum()
    .unstack()
    / 1_000_000_000
)
total_value_year.columns = total_value_year.columns.map(use_code_maps)
total_value_year = total_value_year[total_value_year.columns.sort_values()].round(3)
total_value_year.index = clean_year_index(total_value_year.index)

display(
    style_table(
        total_value_year.reset_index(),
        caption="Total assessed value by property type and roll year",
        formatters={col: fmt_usd_b for col in total_value_year.columns},
    )
)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

median_value_year.plot(ax=ax1, legend=False, linewidth=2)
ax1.set_title("Median Assessed Value by Property Type")
ax1.set_xlabel("Roll Year")
ax1.set_ylabel("Median assessed value")
apply_standard_axis_style(ax1, y_is_currency=True, currency_scale=1)
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x:,.1f}M"))

total_value_year.plot(ax=ax2, legend=False, linewidth=2)
ax2.set_title("Total Assessed Value by Property Type")
ax2.set_xlabel("Roll Year")
ax2.set_ylabel("Total assessed value")
apply_standard_axis_style(ax2, y_is_currency=True, currency_scale=1)
ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"${x:,.0f}B"))

ax2.legend(title="Property type", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

plt.tight_layout()
plt.show()


#### Observations

**At the individual-property level**

- The median assessed value of a **single-family** property rose from **$0.316M in 2007** to **$0.747M in 2024**, an increase of about **136%**.
- The median **multi-family** property rose from **$0.426M** to **$0.994M** over the same period, up roughly **133%**.
- Office and hotel parcels remain expensive on a per-record basis, reaching **$2.2M per Office parcel and $3.0M per Hotel parcel** in 2024. 
- Government-owned vacant parcels frequently have very low or zero assessed values, which is consistent with tax-exempt or administratively different treatment.

**At the aggregate citywide level**

- The total assessed value of **single-family housing** increased from **$58.1B in 2007** to **$154.0B in 2024**.
- **Multi-family housing** increased from **$25.8B** to **$74.7B**, while **office** starts from a very large **$18.4B** base in 2007 to **$59.7B** in 2024 despite having far fewer parcels.
- This distinction is crucial: a category can matter because it is **numerous** (single-family/multi-family), because it is **valuable per parcel** (office/hotel), or both.

This is why the notebook repeatedly separates **median parcel value** from **aggregate tax-base contribution**. The two metrics answer different policy questions.


## 5. Focus section: single-family homes and condominiums

The remainder of the notebook narrows the lens to two especially relevant housing segments:

- **Single-family homes** (`Dwelling`)
- **Condominiums** (`Condominium`)

This narrower focus is useful for downstream housing analysis because these two categories are more comparable to the kinds of properties households actually buy and sell. The notebook filters out records with implausibly small land, improvement, or total assessments so that vacant land, incomplete records, and likely non-comparable cases do not distort neighborhood comparisons.

In [ ]:
def percentile_clip(series: pd.Series, low: float = 0.1, high: float = 0.9) -> tuple[float, float]:
    """Return quantile clip bounds for a numeric series."""
    if isinstance(series, pd.DataFrame):
        series = series.iloc[:, 0]
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return (np.nan, np.nan)
    return (float(s.quantile(low)), float(s.quantile(high)))


def plot_metric_map(
    gdf: gpd.GeoDataFrame,
    metric_col: str,
    title: str,
    cmap: str = "RdYlBu_r",
    center_zero: bool = False,
    clip_quantiles: tuple[float, float] = (0.1, 0.9),
    marker_size: int = 10,
    legend_label: str | None = None,
    ax=None,
    vmin=None,
    vmax=None,
    show_legend: bool = True,
):
    """Plot a building-level metric on top of a lightweight basemap.

    The metric is clipped to selected quantiles to preserve visual contrast for the
    majority of buildings.
    """
    data = gdf.dropna(subset=[metric_col]).copy()
    if data.empty:
        raise ValueError(f"No valid rows available for {metric_col}.")

    q_low, q_high = percentile_clip(data[metric_col], *clip_quantiles)
    clip_low = vmin if vmin is not None else q_low
    clip_high = vmax if vmax is not None else q_high
    data["metric_clipped"] = data[metric_col].clip(clip_low, clip_high)

    data_3857 = data.to_crs(epsg=3857)

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 10))

    if center_zero:
        bound = max(abs(clip_low), abs(clip_high))
        norm = TwoSlopeNorm(vmin=-bound, vcenter=0.0, vmax=bound)
    else:
        norm = Normalize(vmin=clip_low, vmax=clip_high)

    data_3857.plot(
        ax=ax,
        column="metric_clipped",
        cmap=cmap,
        markersize=marker_size,
        alpha=0.9,
        legend=show_legend,
        norm=norm,
        legend_kwds={"shrink": 0.35, "label": legend_label or metric_col},
        zorder=3,
    )

    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, attribution=False)

    ax.set_title(title, fontsize=14)
    ax.set_axis_off()

    xmin, ymin, xmax, ymax = data_3857.total_bounds
    xpad = (xmax - xmin) * 0.05
    ypad = (ymax - ymin) * 0.05
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)

    return norm



# -------------------------------------------------------
# Side-by-side maps with the SAME color scale
# -------------------------------------------------------
def plot_metric_map_dual(
    gdf1,
    gdf2,
    title1 = "San Francisco Single Family\nAssessed Property Value - Year 2024",
    title2 = "San Francisco Condominium\nAssessed Property Value - Year 2024",
    metric_col = "assessed_total",
    cmap = "RdYlBu_r",
    bar_label = "Million USD",
    clip_quantiles = (0.1, 0.9)
):
    # Compute one shared scale across both datasets
    combined_values = pd.concat(
        [
            gdf1[metric_col],
            gdf2[metric_col],
        ],
        ignore_index=True,
    )

    shared_vmin, shared_vmax = percentile_clip(combined_values, *clip_quantiles)

    fig, axes = plt.subplots(1, 2, figsize=(15, 8.5))

    # Plot left map
    norm = plot_metric_map(
        gdf=gdf1,
        metric_col=metric_col,
        title=title1,
        cmap=cmap,
        center_zero=False,
        clip_quantiles=clip_quantiles,
        marker_size=2.2,
        legend_label="",
        ax=axes[0],
        vmin=shared_vmin,
        vmax=shared_vmax,
        show_legend=False,
    )

    # Plot right map
    plot_metric_map(
        gdf=gdf2,
        metric_col=metric_col,
        title=title2,
        cmap=cmap,
        center_zero=False,
        clip_quantiles=clip_quantiles,
        marker_size=2.2,
        legend_label="",
        ax=axes[1],
        vmin=shared_vmin,
        vmax=shared_vmax,
        show_legend=False,
    )

    # Add one shared colorbar for both maps
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axes,
        fraction=0.025,
        pad=0.015,
        aspect=14,
    )
    cbar.set_label(bar_label, fontsize=11)
    cbar.ax.tick_params(labelsize=10)
    cbar.outline.set_linewidth(0.6)

    for ax in axes:
        ax.title.set_fontsize(14)


    # plt.tight_layout()
    plt.show()

Clean up the data for subsequent export

In [ ]:
# Focus on single-family residential records
df_SRES = df.loc[df['use_code'] == 'SRES'].copy()

# Exclude records with implausibly small assessment components (less than $100).
# This helps remove likely vacant/partial records that are not suitable for housing comparison.
df_SRES = df_SRES[df_SRES['assessed_total'] >= 100]
df_SRES = df_SRES[df_SRES['assessed_improvement_value'] >= 100]
df_SRES = df_SRES[df_SRES['assessed_land_value'] >= 100]

# exclude buildings that were 'welfare', 'collge/university', 'religious' etc.
# keeping only 'NA' (unspecified) or '11' (owner occupied)
df_SRES = df_SRES[(df_SRES['exemption_code'].isna()) | (df_SRES['exemption_code']=='11')]

# Building age should be measured relative to the roll year being analyzed, not the current calendar year.
df_SRES['age'] = df_SRES['closed_roll_year'].dt.year - df_SRES['year_property_built']

# Guard against division by zero or missing building area when constructing assessed value per square foot.
valid_area = df_SRES['property_area'].where(df_SRES['property_area'] > 0)
df_SRES['psf'] = df_SRES['assessed_total'] / valid_area


Extract Year 2024 Only

In [ ]:
# Extract Year 2024 data only
df_2024_SRES = df_SRES[df_SRES['closed_roll_year'] == '2024-01-01'].copy()

## add a bit more information
df_2024_SRES['assessor_neighborhood'] = df_2024_SRES['assessor_neighborhood_code'] + ' ' + df_2024_SRES['assessor_neighborhood'] 
df_2024_SRES['assessed_total'] = df_2024_SRES['assessed_total']/10**6

# Keep only the two owner-occupier-oriented classes used in the housing comparison section.
gdf_2024_dwell = gpd.GeoDataFrame(
    df_2024_SRES.loc[df_2024_SRES['property_class_code_definition'] == 'Dwelling'],
    geometry='geometry',
    crs='EPSG:4326'
)
gdf_2024_condo = gpd.GeoDataFrame(
    df_2024_SRES.loc[df_2024_SRES['property_class_code_definition'] == 'Condominium'],
    geometry='geometry',
    crs='EPSG:4326'
)

In [ ]:
# Approximate a recent-sales subset.
# The current cutoff keeps properties sold since 2021, which is close to a trailing 5-year window
# relative to when this notebook was prepared in 2026.
recent_sale_cutoff = pd.Timestamp('2021-01-01')

gdf_2024_condo_5y = gdf_2024_condo[gdf_2024_condo['current_sales_date'] > recent_sale_cutoff].copy()
gdf_2024_dwell_5y = gdf_2024_dwell[gdf_2024_dwell['current_sales_date'] > recent_sale_cutoff].copy()

### 5(A) Where are the more expensive and the more affordable neighborhoods?
### Let's look at Total Assessed Value (Land + Improvement)

In [ ]:
plot_metric_map_dual(
    gdf1 = gdf_2024_dwell,
    gdf2 = gdf_2024_condo,
    title1 = "Single Family\nAssessed Property Value - Year 2024",
    title2 = "Condominium\nAssessed Property Value - Year 2024",
    metric_col = "assessed_total",
    cmap = "RdYlBu_r",
    bar_label = "Million USD",
    clip_quantiles = (0.1, 0.9)
)

In [ ]:
# Neighborhood summary table used for ranking and interpretation
dwelling_neighborhood = (
    gdf_2024_dwell.groupby("assessor_neighborhood")
    .agg(
        median_value=("assessed_total", "median"),
        num_properties=("assessed_total", "size"),
    )
    .sort_values("median_value", ascending=False)
)
dwelling_neighborhood = dwelling_neighborhood[dwelling_neighborhood.num_properties > 100]


In [ ]:
display(
    style_table(
        dwelling_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Property count",
        }),
        caption="Single-family neighborhoods with the highest 2024 median assessed value",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        dwelling_neighborhood.tail(10)[::-1].rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Property count",
        }),
        caption="Single-family neighborhoods with the lowest 2024 median assessed value",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
# Neighborhood summary table used for ranking and interpretation
condo_neighborhood = gdf_2024_condo.groupby('assessor_neighborhood').agg(
    median_value=('assessed_total', 'median'),
    num_properties=('assessed_total', 'size')).sort_values('median_value', ascending=False)
condo_neighborhood = condo_neighborhood[condo_neighborhood.num_properties>100]

In [ ]:
display(
    style_table(
        condo_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Property count",
        }),
        caption="Condominium neighborhoods with the highest 2024 median assessed value",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        condo_neighborhood.tail(10)[::-1].rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Property count",
        }),
        caption="Condominium neighborhoods with the lowest 2024 median assessed value",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Property count": fmt_int,
        },
    )
)


#### Observations

- For **single-family homes**, **Presidio Heights** leads at **$4.729M**, while **Hunters Point** is at **$0.342M**. That is a gap of roughly **13.8x**.
- For **condominiums**, the spread is narrower: **Presidio Heights** leads at **$1.605M**, versus **$0.449M** in **Mount Davidson Manor**, or about **3.6x**.
- That narrower condo spread is an early hint that condo pricing is driven more by **location and building typology**, whereas single-family values combine both **location** and **lot / structure scale**.

### 5(B) What about assessed price per square foot?

Total assessed value is influenced by home size. To normalize across differently sized homes, this section uses **assessed value per square foot**.

This is **not** the same thing as observed market sale price per square foot, but it is still analytically useful because it helps separate two stories:
1. neighborhoods where homes are valuable mainly because they are **large**, and  
2. neighborhoods where each unit of housing space is itself **highly valued**.


In [ ]:
plot_metric_map_dual(
    gdf1 = gdf_2024_dwell,
    gdf2 = gdf_2024_condo,
    title1 = "Single Family\nAssessed Price per Square Foot - Year 2024",
    title2 = "Condominium\nAssessed Price per Square Foot - Year 2024",
    metric_col = "psf",
    cmap = "RdYlBu_r",
    bar_label = "USD",
    clip_quantiles = (0.1, 0.9)
)

In [ ]:
# Neighborhood summary table used for ranking and interpretation
dwelling_neighborhood = gdf_2024_dwell.groupby('assessor_neighborhood').agg(
    median_value=('psf', 'median'),
    num_properties=('psf', 'size')).sort_values('median_value', ascending=False)
dwelling_neighborhood = dwelling_neighborhood[dwelling_neighborhood.num_properties>100]

In [ ]:
display(
    style_table(
        dwelling_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value per sq ft",
            "num_properties": "Property count",
        }),
        caption="Single-family neighborhoods with the highest 2024 assessed value per square foot",
        formatters={
            "Median assessed value per sq ft": fmt_psf,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        dwelling_neighborhood.tail(10)[::-1].rename(columns={
            "median_value": "Median assessed value per sq ft",
            "num_properties": "Property count",
        }),
        caption="Single-family neighborhoods with the lowest 2024 assessed value per square foot",
        formatters={
            "Median assessed value per sq ft": fmt_psf,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
# Neighborhood summary table used for ranking and interpretation
condo_neighborhood = gdf_2024_condo.groupby('assessor_neighborhood').agg(
    median_value=('psf', 'median'),
    num_properties=('psf', 'size')).sort_values('median_value', ascending=False)
condo_neighborhood = condo_neighborhood[condo_neighborhood.num_properties>100]

In [ ]:
display(
    style_table(
        condo_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value per sq ft",
            "num_properties": "Property count",
        }),
        caption="Condominium neighborhoods with the highest 2024 assessed value per square foot",
        formatters={
            "Median assessed value per sq ft": fmt_psf,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        condo_neighborhood.tail(10)[::-1].rename(columns={
            "median_value": "Median assessed value per sq ft",
            "num_properties": "Property count",
        }),
        caption="Condominium neighborhoods with the lowest 2024 assessed value per square foot",
        formatters={
            "Median assessed value per sq ft": fmt_psf,
            "Property count": fmt_int,
        },
    )
)



#### Observations

- For single-family homes, the most expensive space is in **Telegraph Hill ($1,199/sq ft)**, **Pacific Heights ($1,167/sq ft)**, and **Presidio Heights ($1,134/sq ft)**.
- The low end is concentrated in the southeast: **Hunters Point ($253/sq ft)**, **Bayview ($305/sq ft)**, and **Visitacion Valley ($322/sq ft)**.
- For condos, the top tier is much more downtown-oriented: **Financial District South ($1,145/sq ft)**, **Central Waterfront / Dogpatch ($1,145/sq ft)**, **South Beach ($1,142/sq ft)**, and **Mission Bay ($1,047/sq ft)**.
- That pattern says the condo market is pricing access to jobs, waterfront redevelopment, and dense amenity clusters much more directly than the single-family market.

### 5(C) How old are the buildings?

Building age adds an important structural dimension. It can proxy for neighborhood development history, redevelopment pressure, housing form, preservation status, and in some cases prestige or functional obsolescence.

Looking at the age profile helps answer three different questions:

- Which neighborhoods are dominated by **legacy housing stock**?
- Which parts of the city show evidence of **recent development waves**?
- How different is the development history of **single-family districts** versus **condominium-heavy districts**?


In [ ]:
plot_metric_map_dual(
    gdf1 = gdf_2024_dwell,
    gdf2 = gdf_2024_condo,
    title1 = "Single Family\nAge of Building - Year 2024",
    title2 = "Condominium\nAge of Building - Year 2024",
    metric_col = "age",
    cmap = "viridis_r",
    bar_label = "Year",
    clip_quantiles = (0.1, 0.9)
)

In [ ]:
# Neighborhood summary table used for ranking and interpretation
dwelling_neighborhood = gdf_2024_dwell.groupby('assessor_neighborhood').agg(
    median_age_of_building=('age', 'median'),
    num_properties=('age', 'size')).sort_values('median_age_of_building', ascending=False)
dwelling_neighborhood = dwelling_neighborhood[dwelling_neighborhood.num_properties>100]

In [ ]:
display(
    style_table(
        dwelling_neighborhood.head(10).rename(columns={
            "median_age_of_building": "Median building age",
            "num_properties": "Property count",
        }),
        caption="Oldest single-family neighborhoods in the 2024 roll",
        formatters={
            "Median building age": fmt_years,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        dwelling_neighborhood.tail(10)[::-1].rename(columns={
            "median_age_of_building": "Median building age",
            "num_properties": "Property count",
        }),
        caption="Youngest single-family neighborhoods in the 2024 roll",
        formatters={
            "Median building age": fmt_years,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
# Neighborhood summary table used for ranking and interpretation
condo_neighborhood = gdf_2024_condo.groupby('assessor_neighborhood').agg(
    median_age_of_building=('age', 'median'),
    num_properties=('age', 'size')).sort_values('median_age_of_building', ascending=False)
condo_neighborhood = condo_neighborhood[condo_neighborhood.num_properties>100]

In [ ]:
display(
    style_table(
        condo_neighborhood.head(10).rename(columns={
            "median_age_of_building": "Median building age",
            "num_properties": "Property count",
        }),
        caption="Oldest condominium neighborhoods in the 2024 roll",
        formatters={
            "Median building age": fmt_years,
            "Property count": fmt_int,
        },
    )
)


In [ ]:
display(
    style_table(
        condo_neighborhood.tail(10)[::-1].rename(columns={
            "median_age_of_building": "Median building age",
            "num_properties": "Property count",
        }),
        caption="Youngest condominium neighborhoods in the 2024 roll",
        formatters={
            "Median building age": fmt_years,
            "Property count": fmt_int,
        },
    )
)


#### Observations

- The oldest single-family neighborhoods in the table—**Lower Pacific Heights, Hayes Valley, North Panhandle, Inner Mission, and Haight Ashbury**—cluster around a median building age of **124 years**.
- The youngest single-family neighborhood in the filtered table is **Hunters Point** at **36 years**, followed by **Diamond Heights (59 years)** and **Forest Knolls (63 years)**.
- The condo stock is dramatically newer in redevelopment districts: **Hunters Point (8 years)**, **Stonestown (9 years)**, **Central Waterfront/Dogpatch (12 years)**, **Financial District South (15 years)**, and **Mission Bay (15 years)**.
- That contrast tells us the condo market is much more intertwined with **post-2000 redevelopment**, while single-family housing remains predominantly an **older, slower-changing stock**.

### 5(D) Where are the recently sold homes?

Assessed values are most informative about current market conditions when a property has transacted relatively recently. Under California's Proposition 13 system, long-held properties can carry assessed values that remain anchored to an older base-year value and then rise only gradually unless ownership changes or new construction occurs.

This section therefore isolates homes and condos sold within the last five years, then examines their 2024 assessed values. The objective is not to reconstruct exact market sale prices, but to focus on the portion of the roll whose assessments are most likely to be closer to contemporary market conditions.




In [ ]:
plot_metric_map_dual(
    gdf1 = gdf_2024_dwell_5y,
    gdf2 = gdf_2024_condo_5y,
    title1 = "Single Family\nBought & Sold within Last 5 Years\n Assessed Property Value - Year 2024",
    title2 = "Condominium\nBought & Sold within Last 5 Years\n Assessed Property Value - Year 2024",
    metric_col = "assessed_total",
    cmap = "RdYlBu_r",
    bar_label = "Million USD",
    clip_quantiles = (0.1, 0.9)
)

In [ ]:
# Neighborhood summary table used for ranking and interpretation
dwelling_neighborhood = gdf_2024_dwell_5y.groupby('assessor_neighborhood').agg(
    median_value=('assessed_total', 'median'),
    num_properties=('assessed_total', 'size')).sort_values('num_properties', ascending=False)


In [ ]:
display(
    style_table(
        dwelling_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Recent transaction count",
        }),
        caption="Single-family neighborhoods with the highest number of recent transactions",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Recent transaction count": fmt_int,
        },
    )
)


In [ ]:
# Neighborhood summary table used for ranking and interpretation
condo_neighborhood = gdf_2024_condo_5y.groupby('assessor_neighborhood').agg(
    median_value=('assessed_total', 'median'),
    num_properties=('assessed_total', 'size')).sort_values('num_properties', ascending=False)

In [ ]:
display(
    style_table(
        condo_neighborhood.head(10).rename(columns={
            "median_value": "Median assessed value (USD millions)",
            "num_properties": "Recent transaction count",
        }),
        caption="Condominium neighborhoods with the highest number of recent transactions",
        formatters={
            "Median assessed value (USD millions)": fmt_usd_m,
            "Recent transaction count": fmt_int,
        },
    )
)


### 5(E) Owner-occupied vs rental-oriented properties

This final section connects three different lenses:
1. the **size of the housing stock**,  
2. the **recent turnover rate**, and  
3. the **share of properties that appear owner-occupied** using the homeowner exemption flag.

That combination provides a practical way to distinguish:
- neighborhoods dominated by **stable owner households**,  
- neighborhoods with **higher renter or investor presence**, and  
- segments where the market appears to trade more actively.


In [ ]:
summary1 = pd.DataFrame({
    "Units in analysis": [
        gdf_2024_dwell.shape[0],
        gdf_2024_condo.shape[0],
    ],
    "Owner-occupied units": [
        (gdf_2024_dwell.exemption_code == "11").sum(),
        (gdf_2024_condo.exemption_code == "11").sum(),
    ],
    "Owner-occupied share": [
        (gdf_2024_dwell.exemption_code == "11").sum() / gdf_2024_dwell.shape[0] * 100,
        (gdf_2024_condo.exemption_code == "11").sum() / gdf_2024_condo.shape[0] * 100,
    ],
    "Units sold in last 5 years": [
        gdf_2024_dwell_5y.shape[0],
        gdf_2024_condo_5y.shape[0],
    ],
    "5-year turnover rate": [
        gdf_2024_dwell_5y.shape[0] / gdf_2024_dwell.shape[0] * 100,
        gdf_2024_condo_5y.shape[0] / gdf_2024_condo.shape[0] * 100,
    ],
}, index=["Single-family", "Condominium"])

display(
    style_table(
        summary1,
        caption="Occupancy and recent turnover by housing type",
        formatters={
            "Units in analysis": fmt_int,
            "Owner-occupied units": fmt_int,
            "Owner-occupied share": fmt_pct,
            "Units sold in last 5 years": fmt_int,
            "5-year turnover rate": fmt_pct,
        },
    )
)


In [ ]:
summary2 = pd.DataFrame({
    "Units sold in last 5 years": [
        gdf_2024_dwell_5y.shape[0],
        gdf_2024_condo_5y.shape[0],
    ],
    "Recent purchases that are owner-occupied": [
        (gdf_2024_dwell_5y.exemption_code == "11").sum(),
        (gdf_2024_condo_5y.exemption_code == "11").sum(),
    ],
    "Owner-occupied share among recent purchases": [
        (gdf_2024_dwell_5y.exemption_code == "11").sum() / gdf_2024_dwell_5y.shape[0] * 100,
        (gdf_2024_condo_5y.exemption_code == "11").sum() / gdf_2024_condo_5y.shape[0] * 100,
    ],
    "Recent purchases that appear renter-occupied": [
        gdf_2024_dwell_5y.shape[0] - (gdf_2024_dwell_5y.exemption_code == "11").sum(),
        gdf_2024_condo_5y.shape[0] - (gdf_2024_condo_5y.exemption_code == "11").sum(),
    ],
    "Renter-occupied share among recent purchases": [
        (gdf_2024_dwell_5y.shape[0] - (gdf_2024_dwell_5y.exemption_code == "11").sum()) / gdf_2024_dwell_5y.shape[0] * 100,
        (gdf_2024_condo_5y.shape[0] - (gdf_2024_condo_5y.exemption_code == "11").sum()) / gdf_2024_condo_5y.shape[0] * 100,
    ],
}, index=["Single-family", "Condominium"])

display(
    style_table(
        summary2,
        caption="Occupancy mix among units purchased in the last 5 years",
        formatters={
            "Units sold in last 5 years": fmt_int,
            "Recent purchases that are owner-occupied": fmt_int,
            "Owner-occupied share among recent purchases": fmt_pct,
            "Recent purchases that appear renter-occupied": fmt_int,
            "Renter-occupied share among recent purchases": fmt_pct,
        },
    )
)


#### Observations

The results in this section are among the most revealing in the notebook because they show that San Francisco is not operating as one unified housing market. Instead, it appears to function through **Two Housing Circuits**: a **legacy circuit** of tightly held homes, and a **circulation circuit** of homes that re-enter the market and absorb new demand.

- The analysis set contains **94,844 single-family homes** and **52,042 condominiums**.
- The apparent owner-occupied share is **55.6%** for single-family homes, but only **34.2%** for condos. That is a **21.4 percentage-point gap**, suggesting condos play a much larger renter- and investor-facing role.
- Recent turnover is also very different: **8,531 single-family homes** were sold in the last 5 years (**9.0%** of the stock), versus **10,520 condos** (**20.2%** of the stock). In other words, the condo stock is turning over at more than **2.2x** the single-family rate.
- Among recently purchased homes, only **27.8%** of single-family acquisitions and **24.0%** of condo acquisitions appear owner-occupied based on exemption code `11`.
- The remaining shares — **72.2%** for recently purchased single-family homes and **76.0%** for recently purchased condos — appear not owner-occupied.

##### What the numbers imply
- **Single-family homes dominate the stock, but not the active market.** There are far more single-family homes in total, yet fewer of them transact over a five-year period than condos.
- **Condominiums function as circulation housing.** Even though the condo stock is much smaller, it generates more transactions and appears much more exposed to renter and investor demand.
- **Single-family housing looks increasingly like legacy housing.** A majority appears owner-occupied, turnover is low, and a large part of the stock seems to sit outside the normal flow of market exchange.

In other words, the full stock of homes and the actively circulating stock are telling very different stories. San Francisco may contain a large number of homes on paper, but the subset that is realistically available for purchase is much smaller.

##### A particularly important finding: investors appear active in both segments

One of the strongest results in this section is that recently transacted units in **both** property types look overwhelmingly non-owner-occupied after purchase.

That is significant because it cuts against a common simplified narrative in which:

- single-family homes are mainly contested by owner-occupiers, while  
- condos are mainly contested by investors.

The evidence here suggests something more subtle. Once a home enters the **circulation circuit**, both single-family homes and condos appear to face a similar competitive environment. Investors and other non-owner-occupying buyers do **not** seem confined to condos alone; they appear to compete for both types of homes when those homes come onto the market.

A household trying to buy into San Francisco therefore appears to face competition not just in the condo segment, but across the broader active market.

### 5(F) No. of Years Since Last Sale

In [ ]:
gdf_2024_dwell['years_since_last_sale'] = (pd.to_datetime('2024-12-31') - gdf_2024_dwell['current_sales_date']).dt.days / 365.25
gdf_2024_condo['years_since_last_sale'] = (pd.to_datetime('2024-12-31') - gdf_2024_condo['current_sales_date']).dt.days / 365.25

In [ ]:
plot_metric_map_dual(
    gdf1 = gdf_2024_dwell,
    gdf2 = gdf_2024_condo,
    title1 = "Single Family\nNo. of Years Since Last Sale (Year 2024 Data)",
    title2 = "Condominium\nNo. of Years Since Last Sale (Year 2024 Data)",
    metric_col = "years_since_last_sale",
    cmap = "viridis_r",
    bar_label = "Year",
    clip_quantiles = (0.1, 0.9)
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dwell = gdf_2024_dwell['years_since_last_sale'].dropna()
condo = gdf_2024_condo['years_since_last_sale'].dropna()

bins = np.linspace(0, 40, 41)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# --- Dwelling ---
axes[0].hist(dwell, bins=bins)
axes[0].axvline(5, linestyle='--', c="red", linewidth=2)  # <-- FIX
axes[0].set_title('Single-Family Dwellings\nYears Since Last Sale (Y2024 Data)', fontsize=12)
axes[0].set_xlabel('Years')
axes[0].set_ylabel('Count')
axes[0].grid(alpha=0.3)

# --- Condo ---
axes[1].hist(condo, bins=bins)
axes[1].axvline(5, linestyle='--', c="red", linewidth=2)  # <-- ADD HERE TOO
axes[1].set_title('Condominiums\nYears Since Last Sale (Y2024 Data)', fontsize=12)
axes[1].set_xlabel('Years')
axes[1].grid(alpha=0.3)

fig.suptitle('Distribution of Holding Periods (Years Since Last Sale) (Y2024 Data)', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Extract Year 2019 data only
df_2019_SRES = df_SRES[df_SRES['closed_roll_year'] == '2019-01-01'].copy()

# Keep only the two owner-occupier-oriented classes used in the housing comparison section.
gdf_2019_dwell = gpd.GeoDataFrame(
    df_2019_SRES.loc[df_2019_SRES['property_class_code_definition'] == 'Dwelling'],
    geometry='geometry',
    crs='EPSG:4326'
)
gdf_2019_condo = gpd.GeoDataFrame(
    df_2019_SRES.loc[df_2019_SRES['property_class_code_definition'] == 'Condominium'],
    geometry='geometry',
    crs='EPSG:4326'
)

In [ ]:
gdf_2019_dwell['years_since_last_sale'] = (pd.to_datetime('2019-12-31') - gdf_2019_dwell['current_sales_date']).dt.days / 365.25
gdf_2019_condo['years_since_last_sale'] = (pd.to_datetime('2019-12-31') - gdf_2019_condo['current_sales_date']).dt.days / 365.25

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dwell = gdf_2019_dwell['years_since_last_sale'].dropna()
condo = gdf_2019_condo['years_since_last_sale'].dropna()

bins = np.linspace(0, 40, 41)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# --- Dwelling ---
axes[0].hist(dwell, bins=bins)
axes[0].axvline(5, linestyle='--', c="red", linewidth=2)  # <-- FIX
axes[0].set_title('Single-Family Dwellings\nYears Since Last Sale (Y2019 Data)', fontsize=12)
axes[0].set_xlabel('Years')
axes[0].set_ylabel('Count')
axes[0].grid(alpha=0.3)

# --- Condo ---
axes[1].hist(condo, bins=bins)
axes[1].axvline(5, linestyle='--', c="red", linewidth=2)  # <-- ADD HERE TOO
axes[1].set_title('Condominiums\nYears Since Last Sale (Y2019 Data)', fontsize=12)
axes[1].set_xlabel('Years')
axes[1].grid(alpha=0.3)

fig.suptitle('Distribution of Holding Periods (Years Since Last Sale) (Y2019 Data)', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# --- Data ---
dwell = gdf_2019_dwell['years_since_last_sale'].dropna()
condo = gdf_2019_condo['years_since_last_sale'].dropna()

bins = np.linspace(0, 40, 41)

# --- Decay function ---
def exp_decay(x, A, lam):
    return A * np.exp(-lam * x)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, data, title in zip(
    axes,
    [dwell, condo],
    ['Single-Family Dwellings', 'Condominiums']
):
    # Histogram
    counts, edges, _ = ax.hist(data, bins=bins, alpha=1)

    # Prepare for fitting
    x = (edges[:-1] + edges[1:]) / 2
    y = counts

    mask = y > 0
    x_fit = x[mask]
    y_fit = y[mask]

    # Fit decay
    params, _ = curve_fit(exp_decay, x_fit, y_fit, p0=(y.max(), 0.1))
    A, lam = params

    # Smooth curve
    x_smooth = np.linspace(0, 40, 200)
    y_smooth = exp_decay(x_smooth, A, lam)

    ax.plot(x_smooth, y_smooth, linewidth=2, label=f'λ = {lam:.3f}')

    # Styling
    ax.set_title(f'{title}\nYears Since Last Sale (Y2019 Data)', fontsize=12)
    ax.set_xlabel('Years')
    ax.grid(alpha=0.3)
    ax.legend()

# Shared y-label
axes[0].set_ylabel('Count')

# Overall title
fig.suptitle('Decay of Holding Periods: Dwellings vs Condominiums (Y2019 Data)', fontsize=14)

plt.tight_layout()
plt.show()

#### Observations
Section 5(E) established the idea of **Two Housing Circuits** using occupancy and recent turnover. This companion section deepens that interpretation by asking a more dynamic question: **how long do homes stay in the same ownership before returning to the market?**

The key insight from the holding-period figures is that the two housing types do not merely differ in who occupies them today. They differ in the **rhythm of ownership turnover itself**.

#### Main observations from the holding-period figures

- In the 2024 holding-period figures, both property types show a noticeable concentration around homes sold roughly **3 years earlier**, corresponding to the **2021** market. That spike is plausibly related to the unusual pandemic-period housing cycle and should not be treated as the long-run structural norm. Indeed, the spike disappear in the 2019 holding-period figures.
- The condo distribution is much more consistent with a **circulation market**: many units have sold recently, and the number of units falls off steadily as holding periods lengthen. The histogram shows a clear time-depedent exponential decay pattern.
- The single-family distribution is much less consistent with that pattern. Instead of showing a clean market-churn profile, it suggests the presence of a large stock of parcels that remain in the same ownership for very long periods.

#### Decay-function interpretation

The decay-function analysis is especially useful because it captures not just how many homes sold recently, but how quickly ownership cohorts thin out as time passes.

- For **condominiums**, the fitted exponential decay parameter is about **λ = 0.078**.
- Interpreted intuitively, that means the count of condos remaining in a given "not sold since" cohort shrinks by about **7.8% per additional year** in exponential terms.
- This corresponds to an implied **half-life of roughly 8.9 years**.

That is a strong signal of a genuine **circulation circuit**. Condos appear to behave like a housing type that continually recycles through the market. That helps explain why they play such an outsized role in absorbing new demand, setting marginal prices, and reshaping the city's demographic mix.

By contrast:

- For **single-family homes**, the fitted parameter is only about **λ = 0.015**.
- That implies a notional half-life of roughly **46.2 years**, though the fit is weak and should be interpreted cautiously.

The weak fit is itself informative. Single-family homes do **not** appear to follow a clean exponential decay process because a substantial portion of the stock is not behaving like normal circulating inventory at all. Instead, it behaves like **legacy housing**: homes held for decades by the same owners, whether as long-term residences or as long-term rental assets.

#### Why the decay result matters

This decay analysis strengthens the core argument of Section 5(E):

- the condo market is not just **less owner-occupied**; it is also **structurally more recyclable**;
- the single-family market is not just **more owner-occupied**; it is also **structurally less liquid**, with a much larger component of long-duration holding.

This section suggests that the real constraint is not simply the total housing stock, but the size of the **circulation circuit**. That difference is economically important. In a city where prices are set at the margin, the housing problem is driven not only by how many homes exist, but by how many homes are actually **circulating**. The decay curves show that San Francisco's circulation housing is concentrated disproportionately in the condo segment, while the single-family segment contains a large legacy stock that participates only weakly in current market turnover.

A large share of the single-family stock appears effectively **not for sale** in ordinary market conditions. Some of these homes are long-held owner-occupied residences. Others are likely retained as stable rental assets. In both cases, they count toward the city's housing inventory, but they contribute little to current market access.

That helps explain how San Francisco can have a large residential base and still feel extremely difficult to enter as a buyer. The relevant competition is occurring in a much thinner market than the headline stock totals imply.

##### Broader civic interpretation

These results also point to a deeper social balance that cities must manage:

- the **legacy circuit** preserves continuity, neighborhood memory, and long-term rootedness;  
- the **circulation circuit** enables entry, adaptation, and demographic renewal.

A vibrant city requires both. If too much of the stock remains locked in the legacy circuit, the city becomes harder for newcomers to join. If too much of the circulation circuit is captured by investor-oriented demand, the city risks becoming accessible mainly as a place to rent or invest, rather than as a place where new households can put down long-term ownership roots.



#### Final interpretation

Taken together, the results in this notebook are best understood through the framework of **Two Housing Circuits**.

- The **legacy housing circuit** consists of homes that are older, tightly held, and often embedded in long-term owner occupancy or long-term rental holding.
- The **circulation housing circuit** consists of homes that turn over more frequently, absorb new demand, and play a disproportionate role in setting marginal prices and determining who can still enter the city as an owner.

This framework helps connect the value maps, age patterns, turnover tables, and occupancy results into one coherent interpretation.

### 1) The city's value geography is highly unequal

At the neighborhood level, the spread between expensive and affordable areas is very large even within the same property type.

- For single-family homes, median assessed value ranges from about **$4.729M in Presidio Heights** to about **$0.342M in Hunters Point**.
- On a per-square-foot basis, the range runs from about **$1,199/sq ft in Telegraph Hill** to about **$253/sq ft in Hunters Point**.

These are not small deviations around a citywide mean. They reflect fundamentally different submarkets shaped by land scarcity, historical development patterns, neighborhood prestige, redevelopment, and transaction history.

### 2) Single-family homes and condos serve different structural roles

Across the notebook, the contrast between the two property types appears repeatedly:

- **Age:** many single-family neighborhoods are measured in many decades, often around or above a century, while the youngest condo districts are measured in the single digits to teens of years.
- **Turnover:** condos show a **20.2%** five-year turnover rate, versus only **9.0%** for single-family homes.
- **Occupancy:** apparent owner occupancy is **55.6%** for single-family homes, but only **34.2%** for condos.
- **Recent purchase outcomes:** only **27.8%** of recently purchased single-family homes and **24.0%** of recently purchased condos appear owner-occupied.
- **Holding period:** condos show the much clearer circulation pattern, with **λ = 0.078** and an implied half-life of about **8.9 years**, while single-family homes show a weak and much flatter decay pattern, with **λ = 0.015** and a notional half-life of about **46.2 years**.

Together, these findings strongly support the **Two Housing Circuits** interpretation. Single-family homes behave much more like the city's **legacy circuit**. Condos behave much more like the city's **circulation circuit**.

### 3) The active market is much smaller than the total stock

One of the most important findings in the notebook is that **headline housing stock is not the same as accessible market stock**.

- San Francisco contains about **94,844** single-family homes, yet only **8,531** of them sold over the last five years.
- By contrast, the much smaller condo stock of **52,042** units generated **10,520** sales over the same period.

This means that the homes actually setting prices and absorbing new demand come from a much thinner pool than the gross stock counts would suggest. The real battleground for entry is the **circulation circuit**, not the full physical stock of housing.

### 4) Investor-oriented outcomes appear in both property types

A particularly important result is that recently transacted units in **both** segments appear overwhelmingly outside the owner-occupied category based on exemption code `11`.

This suggests that investor or non-owner-occupying demand is not confined to condos. Once a home enters the circulation market, both single-family homes and condos appear to face a similar competitive environment. That is a more nuanced conclusion than the common idea that condos are the investor segment while single-family homes remain primarily a household-buyer segment.

### 5) Proposition 13 and long holding periods shape the assessor data

Assessed values in California reflect ownership history as much as current market conditions. Long-held properties can carry assessment bases far below current market value, while recent sales are much closer to contemporary conditions.

That institutional structure matters for interpretation. It helps explain why recent-sales filtering is so informative, and it also helps explain why a large part of the stock behaves like legacy housing: low turnover is not just a descriptive fact, but part of the economic structure of the market.

### 6) The broader civic issue is continuity versus renewal

The notebook ultimately points to a social and economic tension that goes beyond prices alone.

- The **legacy circuit** preserves continuity, neighborhood memory, and long-term rootedness.
- The **circulation circuit** enables entry, adaptation, and demographic renewal.

A healthy city needs both. If the legacy circuit dominates too completely, the city becomes increasingly closed to newcomers. If the circulation circuit becomes too investor-oriented, the city may remain vibrant as a rental or investment market while becoming less attainable as a place for long-term ownership and household formation.

### 7) Final takeaway

San Francisco's housing challenge is not only a shortage of homes in the abstract. It is also a shortage of homes that are meaningfully **available**, **circulating**, and **accessible** to new owner-occupants.

That is the central lesson of the notebook. The city does not simply contain expensive housing; it contains a large legacy stock that turns over slowly, alongside a much smaller circulation stock that bears most of the pressure of entry, investment demand, and price discovery. Understanding that distinction is essential for interpreting affordability, neighborhood change, and the long-run vitality of the city itself.
